# Filaments: how good could the masks be if finding them were free?

Full-disk H-alpha images of the Sun, 2048 square. Outline every dark filament
**individually**, and no two masks in a frame may share a pixel. Scored by
Panoptic Quality, which is the product of two things: how well the masks are
drawn (SQ) and how well the filaments are found (RQ).

The approach so far marks filament pixels across the whole disk with a U-Net
and takes the connected regions of the thresholded map as the instances. That
reaches **PQ 0.3756** on a frozen validation fold of 142 frames, with **SQ
0.6555** -- against 0.833 between two people outlining the same filament.

Mask quality is the larger of the two shortfalls, and the whole-disk setup has
a ceiling it cannot pass: the network runs at 1024 because 2048 does not fit,
and sending the annotations down to 1024 and back costs them a mean IoU of
0.880 before a network is involved at all.

**This notebook measures whether cutting the frame up would lift that.** One
filament per sample, cut out around its annotated box and resampled to 512 --
four or five times the detail, and a narrower question: not *where are the
filaments* but *draw this one*.

The boxes are the annotated ones, at training and at evaluation. No detector
delivers those, so the result is an **upper bound**: the mask quality
available if finding the filaments were free. That is exactly the number worth
having before building a detector, and it costs one training run rather than a
fortnight.

**How to read it.** The bound is compared against the same annotations drawn
by the whole-disk model.

- If the crop model is far ahead, a two-stage design is worth building, and
  its ceiling is known before any of it exists.
- If it is not, the design is dead and nothing was spent finding out.

Before running: **GPU T4 x2**, **Internet on**, competition data attached as an
input. Save with **Save Version** and set **Save output** under *Advanced
Settings*. About 50 minutes.

## 1. Clone the repository

Cloned rather than pip-installed: `configs/paths.yaml` and the frozen splits in
`configs/splits/` sit beside the package rather than inside it, and a wheel
would leave them behind -- the run would then be validated on a different set
of frames than every other run.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase5-cldice"  # or a commit hash, for a run to reproduce later
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; its CUDA build of torch stays as it is.
!pip install -q segmentation-models-pytorch

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import json
import logging
import os
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import filament
from filament.paths import load_paths

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print("filament", filament.__version__)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )
os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
WORK = Path("/kaggle/working")
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 2. Train on crops

One sample is one annotation rather than one frame, so the 915 annotator-images
of this fold's training side become 6,404 crops. That also makes a small
filament weigh as much as a large one, which no whole-frame loss does: under
Dice a filament of a hundred pixels is a fiftieth of one of five thousand,
while the metric counts them the same.

The input has three channels -- the crop, its contrast-equalised version, and
the box filled in. The third is what says *which* filament is being asked for,
since a crop often contains more than one. It has to be the box rather than
the annotated mask, which would hand over the answer.

The crops sit exactly on their boxes, with no jitter. That is what an upper
bound wants. A system fed by a real detector would need jitter, so that the
seed it trains on is as loose as the one it will be handed.

In [ ]:
from filament.training.config import TrainConfig
from filament.training.loop import train

config = replace(
    TrainConfig.from_yaml(f"{CHECKOUT}/configs/crop_oracle.yaml"),
    num_workers=2,
    output_dir=WORK / "crop_oracle",
)
print(config)
assert config.crops is not None, "This notebook is the crop run; the config says otherwise."

In [ ]:
result = train(config)
print("best epoch", result.best_epoch, "validation loss", round(result.best_val_loss, 4))
print(f"training time: {sum(item.seconds for item in result.history) / 60:.1f} min")

In [ ]:
epochs = [item.epoch for item in result.history]
figure, axes = plt.subplots(figsize=(7, 3.2))
axes.plot(
    epochs,
    [item.train_loss for item in result.history],
    label="train",
    linewidth=2,
    color="#2a78d6",
)
axes.plot(
    epochs,
    [item.val_loss for item in result.history],
    label="validation",
    linewidth=2,
    color="#eb6834",
)
axes.set_xlabel("epoch")
axes.set_ylabel("Dice + cross-entropy")
axes.set_title("Training and validation loss")
axes.legend(frameon=False)
axes.grid(axis="y", linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.show()

## 3. Draw every validation filament from its own box

The mask is compared inside the crop rather than pasted back to 2048: both are
cut the same way, so the comparison is fair, and an upper bound should not
carry losses it can avoid.

Panoptic Quality is deliberately not computed. With the boxes handed over,
finding the filaments is free, and a score including detection would mostly be
measuring the gift.

In [ ]:
from filament.crop_evaluation import score_crops
from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.training.loop import load_checkpoint

model, stored = load_checkpoint(result.checkpoint)
dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(config.fold, f"{CHECKOUT}/configs/splits").val
print(f"fold {config.fold}: {len(val_stems)} validation frames")

crop_result = score_crops(
    model,
    dataset,
    paths.train_images,
    val_stems,
    size=config.crops.size,
    context=config.crops.context,
    seed_padding=config.crops.seed_padding,
    device="cuda",
)
print(crop_result)

## 4. Against the whole-disk model, on the same filaments

The comparison that decides. Both numbers are the IoU each approach reaches on
the same 1,795 annotations of this fold.

The whole-disk figures come from its saved probability maps, scored at the
settings it was tuned with. For an annotation it never touched the IoU is zero,
which is the honest entry: the pipeline had its chance at that filament.

In [ ]:
BASELINE = {
    # The whole-disk model, at the post-processing it was tuned with.
    "pq": 0.3756,
    "sq": 0.6555,
    "rq": 0.5730,
    # Best IoU per annotation, over all 1,795 -- not only the matched ones.
    "mean_iou": 0.4382,
    "median_iou": 0.5314,
    "share_over_half": 0.5393,
}

table = pd.DataFrame(
    [
        {"model": "whole disk, 1024", **{k: BASELINE[k] for k in ("mean_iou", "share_over_half")}},
        {
            "model": "crops, 512, perfect boxes",
            "mean_iou": crop_result.mean_iou,
            "share_over_half": crop_result.share_over_half,
        },
    ]
)
print(table.round(4).to_string(index=False))
print()
print(
    f"mean IoU over every annotation: {BASELINE['mean_iou']:.4f} -> {crop_result.mean_iou:.4f} "
    f"({crop_result.mean_iou - BASELINE['mean_iou']:+.4f})"
)
print(
    f"share reaching IoU 0.5:         {BASELINE['share_over_half']:.1%} -> "
    f"{crop_result.share_over_half:.1%}"
)
print(
    f"mean IoU of those that do:      {BASELINE['sq']:.4f} -> "
    f"{crop_result.mean_iou_over_half:.4f}   (comparable with SQ)"
)

In [ ]:
# Where the gain is, if there is one: small filaments are the ones the
# whole-disk model loses, since a pixel loss barely notices them.
scores = pd.DataFrame([{"area": score.area, "iou": score.iou} for score in crop_result.scores])
bands = [0, 400, 800, 1600, 3200, 10**9]
labels = ["<400", "400-800", "800-1.6k", "1.6k-3.2k", ">3.2k"]
scores["band"] = pd.cut(scores["area"], bins=bands, labels=labels, right=False)
by_band = scores.groupby("band", observed=True).agg(
    annotations=("iou", "size"),
    mean_iou=("iou", "mean"),
    share_over_half=("iou", lambda values: float((values > 0.5).mean())),
)
# What the whole-disk model reaches in the same bands.
by_band["whole_disk_share"] = [0.157, 0.425, 0.592, 0.657, 0.652]
print(by_band.round(3).to_string())

In [ ]:
figure, axes = plt.subplots(figsize=(6.4, 3.6))
positions = np.arange(len(by_band))
axes.bar(
    positions - 0.2, by_band["whole_disk_share"], width=0.4, label="whole disk", color="#b8b5ad"
)
axes.bar(
    positions + 0.2,
    by_band["share_over_half"],
    width=0.4,
    label="crops, perfect boxes",
    color="#2a78d6",
)
axes.set_xticks(positions)
axes.set_xticklabels(by_band.index, fontsize=8)
axes.set_xlabel("annotation area, pixels at 2048")
axes.set_ylabel("share reaching IoU 0.5")
axes.set_title("Where the masks are good enough to count")
axes.legend(frameon=False)
axes.grid(axis="y", linewidth=0.8, color="#e3e2dd")
axes.set_axisbelow(True)
axes.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(WORK / "by_area.png", dpi=140)
plt.show()

## 5. Look at a few

Numbers say how far the masks are from the annotations; they do not say what
the remaining difference looks like. The worst cases are the ones to read: a
filament drawn far too short is a different problem from one drawn with a
ragged edge.

In [ ]:
from filament.crop_evaluation import draw_from_box
from filament.data.crops import Box, build_input, cut
from filament.data.image import load_grayscale

by_stem = dataset.by_stem()
ordered = sorted(crop_result.scores, key=lambda score: score.iou)
picked = [ordered[0], ordered[len(ordered) // 2], ordered[-1]]

figure, axes = plt.subplots(len(picked), 3, figsize=(9, 3.2 * len(picked)), squeeze=False)
for row, score in enumerate(picked):
    entry = next(e for e in by_stem[score.stem] if e.image_id == score.image_id)
    annotation = next(a for a in entry.annotations if str(a.annotation_id) == score.annotation_id)
    truth = annotation.to_mask(entry.height, entry.width)
    rows_, columns_ = np.flatnonzero(truth.any(axis=1)), np.flatnonzero(truth.any(axis=0))
    box = Box(int(rows_[0]), int(columns_[0]), int(rows_[-1]) + 1, int(columns_[-1]) + 1)

    frame = load_grayscale(paths.train_images / entry.file_name)
    predicted, window = draw_from_box(
        model,
        frame,
        box,
        config.crops.size,
        config.crops.context,
        config.crops.seed_padding,
        device="cuda",
    )
    prepared = build_input(frame, box, window, config.crops.size)
    reference = cut(truth.astype(np.uint8), window, config.crops.size, mask=True).astype(bool)

    for column, (image, title, cmap) in enumerate(
        [
            (prepared[1], "what the model sees", "gray"),
            (reference, "annotated", "gray"),
            (predicted, "drawn", "gray"),
        ]
    ):
        axis = axes[row][column]
        axis.imshow(image, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
        axis.set_xticks([])
        axis.set_yticks([])
        if row == 0:
            axis.set_title(title)
    axes[row][0].set_ylabel(f"IoU {score.iou:.3f}\n{score.area} px", fontsize=8)
plt.tight_layout()
plt.savefig(WORK / "examples.png", dpi=140)
plt.show()

## 6. Package the result

Small enough to carry off the machine: the numbers, the two figures, and the
per-annotation scores, which is what the analysis deciding the next step runs
off. The checkpoint stays in the notebook output.

In [ ]:
import zipfile

summary = {
    "commit": REF,
    "crops": crop_result.to_dict(),
    "baseline": BASELINE,
    "by_area": by_band.round(4).reset_index().to_dict(orient="records"),
    "best_epoch": result.best_epoch,
    "best_val_loss": result.best_val_loss,
    "training_minutes": sum(item.seconds for item in result.history) / 60,
    "config": config.to_dict(),
}
(WORK / "crop_oracle_summary.json").write_text(json.dumps(summary, indent=2, default=str))
pd.DataFrame(
    [
        {
            "stem": score.stem,
            "image_id": score.image_id,
            "annotation_id": score.annotation_id,
            "area": score.area,
            "iou": round(score.iou, 4),
        }
        for score in crop_result.scores
    ]
).to_csv(WORK / "crop_scores.csv", index=False)

bundle = WORK / "crop_oracle_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for name in ("crop_oracle_summary.json", "crop_scores.csv", "by_area.png", "examples.png"):
        archive.write(WORK / name, arcname=name)

print(f"{bundle.name}: {bundle.stat().st_size / 1e6:.1f} MB")
print(json.dumps(summary["crops"], indent=2))

## 7. Reading the result

**The mean IoU over every annotation** is the headline: the whole-disk model
reaches 0.4382, counting zero for the filaments it never found. The crop model
is handed those filaments, so a large gap is expected -- what matters is
whether the gap is large enough to pay for a detector that would have to find
them.

**The share reaching IoU 0.5** is what the score actually gates on. A run that
lifts the mean without moving that share would change nothing.

**The mean IoU among those clearing 0.5** is the one comparable with SQ 0.6555.
If it lands near 0.66, then drawing from a perfect box is no better than what
the whole-disk model already does, and the two-stage design has nothing to
offer however good the detector is.

The per-area table says where any gain sits. The whole-disk model reaches IoU
0.5 on 15.7% of annotations under 400 pixels and on 65% of those above 3,200 --
if the crop model closes that gap, it is doing what a pixel-weighted loss
cannot.